In [1]:
import time
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [2]:
spark = SparkSession.builder \
    .appName("Benchmark_Optimisations_Avancees_N5_N10") \
    .master("spark://spark-master:7077") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "1") \
    .config("spark.sql.warehouse.dir", "hdfs://namenode:9000/user/hive/warehouse") \
    .getOrCreate()

#### <span style="background-color: yellow;">NIVEAU 5 : PARTITION PRUNING (Partitionnement physique sur HDFS)</span>

In [3]:
# Optimisation : On écrit les données en les découpant physiquement par dossier 'Year'.
# Quand on filtre sur une année précise (ex: Year == 2005), Spark ignore 95% des dossiers sur HDFS.
print("Écriture avec Partitionnement physique par Year...")
spark.read.parquet("hdfs://namenode:9000/data_parquet/flights") \
    .write.mode("overwrite").partitionBy("Year") \
    .parquet("hdfs://namenode:9000/data_partitioned/flights")

Écriture avec Partitionnement physique par Year...


In [4]:
def run_partition_pruning():
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_partitioned/flights") \
        .filter((F.col("Year") == 2005) & (F.col("Cancelled") == 0))
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    df_carriers = spark.read.parquet("hdfs://namenode:9000/data_parquet/carriers")
    
    return df_flights \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .join(F.broadcast(df_carriers), df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(F.count("*").alias("total_vols"), F.avg("ArrDelay").alias("retard_moyen"))

In [6]:
results = {}
def mesurer_requete(nom_etape, fonction_requete):
    print(f"--- Exécution : {nom_etape} ---")
    debut = time.time()
    df_res = fonction_requete()
    # Action (.count() ou .collect()) pour forcer Spark à exécuter le calcul (Évaluation Paresseuse / Lazy Evaluation)
    lignes = df_res.count() 
    fin = time.time()
    duree = round(fin - debut, 2)
    results[nom_etape] = duree
    print(f"✅ Terminé en {duree} secondes ({lignes} lignes générées)\n")
    return df_res


In [7]:
mesurer_requete("5. Partition Pruning (Filtre Year=2005)", run_partition_pruning)

--- Exécution : 5. Partition Pruning (Filtre Year=2005) ---
✅ Terminé en 5.84 secondes (407995 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen: double]

#### <span style="background-color: yellow;">NIVEAU 6 : FONCTIONS NATIVES SPARK vs UDF PYTHON (Sérialisation)</span>

In [8]:
# Optimisation : Les UDF Python obligent Spark à sortir de la JVM et sérialiser les données vers Python (très lent).
# On compare une règle métier complexe avec une fonction native Spark (F.when).
def run_native_vs_udf():
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights") \
        .filter(F.col("Cancelled") == 0)
    
    # Utilisation de F.when (exécuté directement en bytecode Java/Tungsten, 0 sérialisation Python)
    df_categorized = df_flights.withColumn(
        "statut_retard",
        F.when(F.col("ArrDelay") <= 0, "A l'heure")
         .when((F.col("ArrDelay") > 0) & (F.col("ArrDelay") <= 30), "Retard Mineur")
         .otherwise("Retard Majeur")
    )
    return df_categorized.groupBy("statut_retard").agg(F.count("*").alias("nb_vols"))

In [9]:
mesurer_requete("6. Catalyst & Tungsten (Native Expressions vs UDF)", run_native_vs_udf)

--- Exécution : 6. Catalyst & Tungsten (Native Expressions vs UDF) ---
✅ Terminé en 2.18 secondes (3 lignes générées)



DataFrame[statut_retard: string, nb_vols: bigint]

#### <span style="background-color: yellow;"># NIVEAU 7 : ADAPTIVE QUERY EXECUTION (AQE - Spark 3.x)</span>

In [11]:
# Optimisation : AQE ajuste dynamiquement le plan d'exécution pendant le runtime 
# (fusionne les petites partitions post-shuffle et optimise les joins dynamiquement).
def run_aqe():
    spark.conf.set("spark.sql.adaptive.enabled", "true")
    spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
    spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
    
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights").filter(F.col("Cancelled") == 0)
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    df_carriers = spark.read.parquet("hdfs://namenode:9000/data_parquet/carriers")
    
    return df_flights \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .join(F.broadcast(df_carriers), df_flights.carrier_id == df_carriers.carrier_id) \
        .groupBy(df_airports.ville, df_carriers.nom) \
        .agg(F.count("*").alias("total_vols"), F.avg("ArrDelay").alias("retard_moyen"))

In [12]:
mesurer_requete("7. Adaptive Query Execution (AQE Activé)", run_aqe)

--- Exécution : 7. Adaptive Query Execution (AQE Activé) ---
✅ Terminé en 12.02 secondes (3752980 lignes générées)



DataFrame[ville: string, nom: string, total_vols: bigint, retard_moyen: double]

#### <span style="background-color: yellow;">NIVEAU 8 : GESTION DU DATA SKEW (Technique du Salting)</span>

In [13]:
# Problème simulé : 80% des vols partent d'un seul aéroport (origin_id = 1), créant un goulot d'étranglement sur 1 worker.
# Optimisation : On ajoute un "Sel" (clé aléatoire 0 à 3) pour éclater la clé skewée sur tous les workers.
def run_salting_skew():
    df_flights = spark.read.parquet("hdfs://namenode:9000/data_parquet/flights")
    
    # Simulation d'un fort Data Skew (80% sur origin_id = 1)
    df_skewed = df_flights.withColumn(
        "skewed_origin", F.when(F.rand() < 0.8, F.lit(1)).otherwise(F.col("origin_id"))
    )
    
    # Application du Salting (ajout d'un suffixe aléatoire de 0 à 3)
    SALT_FACTOR = 4
    df_salted = df_skewed.withColumn("salt", (F.rand() * SALT_FACTOR).cast("int"))
    
    # Double agrégation (Locale avec le sel, puis Globale sans le sel)
    etape1 = df_salted.groupBy("skewed_origin", "salt").agg(F.sum("ArrDelay").alias("sum_delay"), F.count("*").alias("cnt"))
    etape2 = etape1.groupBy("skewed_origin").agg((F.sum("sum_delay") / F.sum("cnt")).alias("avg_delay"))
    return etape2

In [14]:
mesurer_requete("8. Résolution Data Skew (Technique du Salting)", run_salting_skew)

--- Exécution : 8. Résolution Data Skew (Technique du Salting) ---
✅ Terminé en 1.18 secondes (3400 lignes générées)



DataFrame[skewed_origin: int, avg_delay: double]

#### <span style="background-color: yellow;">NIVEAU 9 : FORMAT ORC + COMPRESSION ZSTD</span>

In [20]:
# Optimisation : Comparaison de Parquet (Snappy) avec ORC (Optimized Row Columnar) compressé en Zstandard (Zstd)
# pour maximiser le ratio compression / vitesse de lecture I/O sur HDFS.
print("Écriture des données en ORC + Zstd...")
spark.read.parquet("hdfs://namenode:9000/data_parquet/flights") \
    .write.mode("overwrite").option("compression", "zstd") \
    .orc("hdfs://namenode:9000/data_orc/flight")

Écriture des données en ORC + Zstd...


In [23]:
def run_orc_zstd():
    df_flights = spark.read.orc("hdfs://namenode:9000/data_orc/flight").filter(F.col("Cancelled") == 0)
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    
    return df_flights \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .groupBy(df_airports.ville) \
        .agg(F.count("*").alias("total_vols"), F.avg("ArrDelay").alias("retard_moyen"))


In [24]:
mesurer_requete("9. Format ORC + Compression Zstd", run_orc_zstd)

--- Exécution : 9. Format ORC + Compression Zstd ---
✅ Terminé en 3.27 secondes (3025 lignes générées)



DataFrame[ville: string, total_vols: bigint, retard_moyen: double]

#### <span style="background-color: yellow;">NIVEAU 10 : BUCKETING (Pré-tri et pré-hachage sur disque)</span>

In [25]:
# Optimisation : On sauvegarde la table triée et hachée en 8 buckets sur 'origin_id'.
# Élimine complètement le Shuffle lors des opérations de jointure ou d'agrégation répétées sur cette clé.
print("Création de la table Bucketed...")
spark.read.parquet("hdfs://namenode:9000/data_parquet/flights") \
    .write.mode("overwrite") \
    .bucketBy(8, "origin_id") \
    .sortBy("origin_id") \
    .saveAsTable("flights_bucketed")

Création de la table Bucketed...


In [26]:
def run_bucketing():
    df_flights = spark.table("flights_bucketed").filter(F.col("Cancelled") == 0)
    df_airports = spark.read.parquet("hdfs://namenode:9000/data_parquet/airports")
    
    return df_flights \
        .join(F.broadcast(df_airports), df_flights.origin_id == df_airports.airport_id) \
        .groupBy(df_airports.ville) \
        .agg(F.count("*").alias("total_vols"), F.avg("ArrDelay").alias("retard_moyen"))

mesurer_requete("10. Bucketing (Pre-hashed & Sorted Table)", run_bucketing)

--- Exécution : 10. Bucketing (Pre-hashed & Sorted Table) ---
✅ Terminé en 2.09 secondes (3025 lignes générées)



DataFrame[ville: string, total_vols: bigint, retard_moyen: double]

In [27]:
spark.catalog.clearCache()

In [28]:
import json

fichier_json = "/home/jovyan/work/benchmark_results.json"

# 1. Lire les résultats précédents (N1-N4)
try:
    with open(fichier_json, "r") as f:
        anciens_results = json.load(f)
except FileNotFoundError:
    anciens_results = {}

# 2. Fusionner (Merge) avec les nouveaux résultats (N5-N10)
anciens_results.update(results)

# 3. Réécrire l'ensemble dans le même fichier
with open(fichier_json, "w") as f:
    json.dump(anciens_results, f, indent=4)

print("✅ Tous les résultats (N1 à N10) sont réunis dans benchmark_results.json !")
print(json.dumps(anciens_results, indent=4))

✅ Tous les résultats (N1 à N10) sont réunis dans benchmark_results.json !
{
    "1. R\u00e9f\u00e9rence (CSV + SortMergeJoin)": 56.64,
    "2. Format Parquet (Column Pruning)": 15.25,
    "3. Parquet + Broadcast Join": 12.25,
    "4. Parquet + Broadcast + Tuning Partitions/Cache": 26.48,
    "5. Partition Pruning (Filtre Year=2005)": 5.84,
    "6. Catalyst & Tungsten (Native Expressions vs UDF)": 2.18,
    "7. Adaptive Query Execution (AQE Activ\u00e9)": 12.02,
    "8. R\u00e9solution Data Skew (Technique du Salting)": 1.18,
    "9. Format ORC + Compression Zstd": 3.27,
    "10. Bucketing (Pre-hashed & Sorted Table)": 2.09
}


In [29]:
spark.stop()